In [5]:
import sys
import os
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# IPIE / PySCF Imports
from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler
import ipie.estimators.local_energy_sd

# Check for GPU (CuPy) support
try:
    import cupy as cp
    has_cupy = True
except ImportError:
    cp = None
    has_cupy = False

# =============================================================================
# 0. CONFIGURATION & ML LOADING
# =============================================================================
N_ATOMS = 10
TEST_SEED = 999 
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
DEPLOY_DIR = "deployment_objects"
MODEL_PATH = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

# =============================================================================
# 1. HELPER FUNCTIONS
# =============================================================================
def get_dynamic_operators(mol):
    S = mol.intor('int1e_ovlp')
    h_core_ao = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
    e, v = np.linalg.eigh(S)
    mask = e > 1e-15
    S_inv_sqrt = v[:, mask] @ np.diag(e[mask]**(-0.5)) @ v[:, mask].T
    S_sqrt = v[:, mask] @ np.diag(e[mask]**(0.5)) @ v[:, mask].T
    h_core_lowdin = S_inv_sqrt.T @ h_core_ao @ S_inv_sqrt
    return S_inv_sqrt, S_sqrt, h_core_lowdin, S

# =============================================================================
# 2. THE ML PROXY (Device-Aware Version)
# =============================================================================
def create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, 
    P_hf_ref, E_hf_ref, S_sqrt, h_core_dyn, C_a, 
    use_gpu, n_atoms
):
    # Ensure constants are NumPy for CPU-based algebra
    P_hf_ref_np = np.asarray(P_hf_ref)
    S_sqrt_np = np.asarray(S_sqrt)
    h_core_dyn_np = np.asarray(h_core_dyn)
    C_a_np = np.asarray(C_a)

    @tf.function(reduce_retracing=True)
    def fast_predict(inputs):
        return ml_model(inputs, training=False)

    call_counter = {"count": 0}

    def local_energy_single_det_uhf(system, hamiltonian, walkers, trial):
        if call_counter["count"] < 1:
            print(f"\n[ML-PROXY] Intercepted call. CPU Algebra -> Device-Aware Casting.")
            call_counter["count"] += 1

        nwalkers = walkers.nwalkers
        nalpha = trial.nalpha
        
        # 1. Safely pull walker data to CPU (Standard NumPy)
        def to_cpu(obj):
            return obj.get() if hasattr(obj, 'get') else obj

        # IPIE UHFWalkers use phia/phib; handle fallback for GHF/single phi
        phi_a = to_cpu(walkers.phia) if hasattr(walkers, 'phia') else to_cpu(walkers.phi[:, :, :nalpha])
        phi_b = to_cpu(walkers.phib) if hasattr(walkers, 'phib') else to_cpu(walkers.phi[:, :, nalpha:])
        Psi_T_a = to_cpu(trial.psi[:, :nalpha])
        Psi_T_b = to_cpu(trial.psi[:, nalpha:])

        # 2. 1-RDM Construction on CPU (Prevents libnvrtc errors)
        O_a = np.einsum('ui, wuj -> wij', Psi_T_a.conj(), phi_a)
        O_b = np.einsum('ui, wuj -> wij', Psi_T_b.conj(), phi_b)
        invO_a = np.linalg.inv(O_a)
        invO_b = np.linalg.inv(O_b)
        
        G_mo_a = np.einsum('wvi, wiu -> wvu', phi_a, np.einsum('wij, ju -> wiu', invO_a, Psi_T_a.conj().T))
        G_mo_b = np.einsum('wvi, wiu -> wvu', phi_b, np.einsum('wij, ju -> wiu', invO_b, Psi_T_b.conj().T))
        
        # [FIXED] Corrected tensor contraction for AO transformation mapping 
        P_ao = (np.einsum("qi, wij, pj -> wqp", C_a_np, G_mo_a, C_a_np.conj()) + 
                np.einsum("qi, wij, pj -> wqp", C_a_np, G_mo_b, C_a_np.conj()))

        # 3. Feature Engineering & ML Inference
        P_lowdin = np.einsum('ai, wib, bj -> waj', S_sqrt_np, P_ao, S_sqrt_np)
        delta_P = P_lowdin - P_hf_ref_np
        E_1B_delta = np.einsum('ij, wji -> w', h_core_dyn_np, delta_P).real

        X_input = np.stack([np.real(delta_P), np.imag(delta_P)], axis=-1).astype(np.float32)
        X_flat = X_input.reshape(nwalkers, -1)
        X_scaled = X_scaler.transform(X_flat).reshape(nwalkers, n_atoms, n_atoms, 2)

        preds_scaled = fast_predict(X_scaled).numpy()
        E_corr_delta = y_scaler.inverse_transform(preds_scaled).flatten()

        # 4. Final Energy Construction
        energy_out = np.zeros((nwalkers, 3), dtype=np.complex128)
        energy_out[:, 0] = E_hf_ref + E_1B_delta + E_corr_delta
        energy_out[:, 1] = E_hf_ref + E_1B_delta
        energy_out[:, 2] = E_corr_delta
        
        # --- THE FIX: DEVICE-AWARE RETURN ---
        # Detect if walkers are living on GPU (CuPy)
        is_gpu_walker = hasattr(walkers.weight, 'device') or 'cupy' in str(type(walkers.weight))
        print("Here DL")
        if is_gpu_walker and cp is not None:
            return cp.asarray(energy_out)
        return energy_out

    return local_energy_single_det_uhf

# =============================================================================
# 3. MAIN INITIALIZATION & PATCHING
# =============================================================================
comm = MPIHandler()
rank = comm.rank

# [FIXED] Load ML assets universally across all MPI ranks
if rank == 0:
    print(f">>> LOADING ML ASSETS...")

try:
    ml_model = load_model(MODEL_PATH, custom_objects={"log_cosh_loss": log_cosh_loss})
    X_scaler = joblib.load(os.path.join(DEPLOY_DIR, "X_scaler.save"))
    y_scaler = joblib.load(os.path.join(DEPLOY_DIR, "y_scaler.save"))
    P_hf_ref = np.load(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"))
except Exception as e:
    if rank == 0:
        print(f"Error loading ML assets: {e}")
    sys.exit(1)

if rank == 0:
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.kernel()
    gen_ipie_input_from_pyscf_chk(mf.chkfile, hamil_file=f"ham_h{N_ATOMS}.h5", wfn_file=f"wfn_h{N_ATOMS}.h5", verbose=0)
    
    E_HF = mf.e_tot
    C_a = mf.mo_coeff[0] if np.ndim(mf.mo_coeff) == 3 else mf.mo_coeff
    _, S_sqrt, h_core, _ = get_dynamic_operators(mol)
else:
    E_HF = C_a = S_sqrt = h_core = None

# Broadcast PySCF setup to all ranks
E_HF = comm.comm.bcast(E_HF, root=0)
C_a = comm.comm.bcast(C_a, root=0)
S_sqrt = comm.comm.bcast(S_sqrt, root=0)
h_core = comm.comm.bcast(h_core, root=0)

# Build AFQMC object
afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2), ham_file=f"ham_h{N_ATOMS}.h5",
    wfn_file=f"wfn_h{N_ATOMS}.h5", num_walkers=WALKERS, num_blocks=50,    num_steps_per_block=1, verbose=0,seed=TEST_SEED 
)
afqmc.mpi_handler = comm

# Create the patch function
ml_proxy = create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, P_hf_ref, E_HF, S_sqrt, h_core, C_a, has_cupy, N_ATOMS
)

# Global Patching: Replace 'local_energy_single_det_uhf' wherever it is imported
original_func = ipie.estimators.local_energy_sd.local_energy_single_det_uhf
for mod_name, module in list(sys.modules.items()):
    if module is None or not mod_name.startswith("ipie"): continue
    try:
        for attr_name, attr_value in vars(module).items():
            if attr_value is original_func:
                setattr(module, attr_name, ml_proxy)
    except: pass

# Instance Patching: For the current simulation session
if hasattr(afqmc, 'propagator'):
    afqmc.propagator.local_energy = ml_proxy

# =============================================================================
# 4. RUN
# =============================================================================
if rank == 0: 
    print("\n" + "#"*60 + "\n### STARTING ML-AFQMC PRODUCTION RUN ###\n" + "#"*60)

afqmc.run()

>>> LOADING ML ASSETS...
# random seed is 999

############################################################
### STARTING ML-AFQMC PRODUCTION RUN ###
############################################################
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body

[ML-PROXY] Intercepted call. CPU Algebra -> Device-Aware Casting.
Here DL
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -1.0441186452672033e+04  2.0480000000000000e+03 -5.0982355725937660e+00 -5.0965095726060818e+00 -1.7259999876841903e-03
Here DL
                1   2.0616780949217637e+03  2.0480000000000000e+03 -2.6630942934985447e+00 -1.0512960377430751e+04  2.0616780949217637e+03 -5.0992249485144265e+00 -5.0965077286629601e+00 -2.7172198514668227e-03
Here DL
                2   2.0617445384773637e+03  2.0480000000000000e+03

In [6]:
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable
qmc_data = extract_observable(afqmc.estimators.filename, "energy")
y = qmc_data["ETotal"][1:]
df = reblock_by_autocorr(y, verbose=1)
print(df)

# Reblock based on autocorrelation time
nsamples, tac = 3, -1.1102230246251565e-16
nsamples, tac = 6, 0.7156472381353625
nsamples, tac = 12, 1.1199714070785962
nsamples, tac = 25, 3.4161188649724297
nsamples, tac = 50, 7.135664511630411
   ETotal_ac  ETotal_error_ac  ETotal_nsamp_ac  ac
0   -5.11874         0.004293                6   8


In [2]:
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable
qmc_data = extract_observable(afqmc.estimators.filename, "energy")
y = qmc_data["ETotal"][1:]
df = reblock_by_autocorr(y, verbose=1)
print(df)

# Reblock based on autocorrelation time
nsamples, tac = 3, 0.0
nsamples, tac = 6, 0.7158418934217836
nsamples, tac = 12, 1.118805351105539
nsamples, tac = 25, 3.409580224412455
nsamples, tac = 50, 7.130544699165089
   ETotal_ac  ETotal_error_ac  ETotal_nsamp_ac  ac
0  -5.118546         0.004539                6   8


In [ ]:
afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2),
    ham_file=ham_file,
    wfn_file=wfn_file,
    num_blocks=50,              
    num_steps_per_block=1,
    num_walkers=WALKERS,
    seed=999,             
    verbose=0
)

In [5]:
-5.119075--5.118546

-0.0005289999999993356